In [ ]:
import os, random
import numpy as np
import pandas as pd
from pathlib import Path

import soundfile as sf
import librosa

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
XGB_AVAILABLE = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from scipy.stats import randint, uniform

In [ ]:
!pip install -q wandb
import wandb
wandb.login()
WANDB_PROJECT = "speech-defects-final"
WANDB_GROUP = "baselines"
WANDB_ENABLED = True

def start_baseline_run(name, config):
    if not WANDB_ENABLED:
        return None
    run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=name,
        config=config,
        reinit=True,
    )
    return run

def finish_baseline_run(run):
    if run is not None:
        run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aksenovmr (aksenovmr-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
def metrics_dict(y_true, y_pred, y_score_bad, thr):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "thr": float(thr),
        "auc": float(roc_auc_score(y_true, y_score_bad)),
        "ap": float(average_precision_score(y_true, y_score_bad)),
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="binary", pos_label=1)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "precision": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
    }

def log_baseline_metrics(
    run,
    val_default,
    val_best,
    test_default,
    test_best,
    thr_bad,
    extra=None,
):
    if run is None:
        return

    payload = {
        "thresholds/thr_bad": float(thr_bad),

        "val/default_auc": val_default["auc"],
        "val/default_ap": val_default["ap"],
        "val/default_acc": val_default["acc"],
        "val/default_f1": val_default["f1"],
        "val/default_f1_macro": val_default["f1_macro"],
        "val/default_precision": val_default["precision"],
        "val/default_recall": val_default["recall"],

        "val/best_f1_auc": val_best["auc"],
        "val/best_f1_ap": val_best["ap"],
        "val/best_f1_acc": val_best["acc"],
        "val/best_f1_f1": val_best["f1"],
        "val/best_f1_f1_macro": val_best["f1_macro"],
        "val/best_f1_precision": val_best["precision"],
        "val/best_f1_recall": val_best["recall"],

        "test/default_auc": test_default["auc"],
        "test/default_ap": test_default["ap"],
        "test/default_acc": test_default["acc"],
        "test/default_f1": test_default["f1"],
        "test/default_f1_macro": test_default["f1_macro"],
        "test/default_precision": test_default["precision"],
        "test/default_recall": test_default["recall"],

        "test/best_f1_from_val_auc": test_best["auc"],
        "test/best_f1_from_val_ap": test_best["ap"],
        "test/best_f1_from_val_acc": test_best["acc"],
        "test/best_f1_from_val_f1": test_best["f1"],
        "test/best_f1_from_val_f1_macro": test_best["f1_macro"],
        "test/best_f1_from_val_precision": test_best["precision"],
        "test/best_f1_from_val_recall": test_best["recall"],
    }

    if extra:
        payload.update(extra)

    wandb.log(payload)

In [ ]:
GOOD_ZIP = "/content/хорошо.zip"
BAD_ZIP  = "/content/плохо.zip"

GOOD_DIR = Path("/content/хорошо")
BAD_DIR  = Path("/content/плохо")

GOOD_DIR.mkdir(parents=True, exist_ok=True)
BAD_DIR.mkdir(parents=True, exist_ok=True)

!ls -lh "/content"
!file "{GOOD_ZIP}"
!file "{BAD_ZIP}"

!apt-get -qq update
!apt-get -qq install -y p7zip-full

!7z x "{GOOD_ZIP}" -o"{GOOD_DIR}" -y
!7z x "{BAD_ZIP}"  -o"{BAD_DIR}"  -y

good_wavs = sorted(GOOD_DIR.rglob("*.wav"))
bad_wavs  = sorted(BAD_DIR.rglob("*.wav"))

total 662M
drwxr-xr-x 1 root root 4.0K Apr 16 13:33 sample_data
drwxr-xr-x 2 root root 4.0K May  7 08:52 плохо
-rw-r--r-- 1 root root 274M May  7 08:48 плохо.zip
drwxr-xr-x 2 root root 4.0K May  7 08:52 хорошо
-rw-r--r-- 1 root root 389M May  7 08:50 хорошо.zip
/content/хорошо.zip: Zip archive data, at least v2.0 to extract, compression method=deflate
/content/плохо.zip: Zip archive data, at least v2.0 to extract, compression method=deflate
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/                   1 file, 406858082 bytes (389 MiB)

Extracting archive: /con

In [ ]:
seed = 42
rows=[]
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [ ]:
df = pd.DataFrame({
    "path": [str(p) for p in good_wavs] + [str(p) for p in bad_wavs],
    "label": [0]*len(good_wavs) + [1]*len(bad_wavs)})
print("\ndf shape:", df.shape)
df.sample(5, random_state=seed)


df shape: (2775, 2)


,path,label
879,/content/хорошо/eu.80be7cf3-5846-4608-981e-c04...,0
1989,/content/плохо/eu.1f5bfae8-63cf-48ec-bfd3-cdf1...,1
889,/content/хорошо/eu.81906517-c46d-4a8e-8827-d68...,0
912,/content/хорошо/eu.84aff07d-e9b6-45df-a229-3e3...,0
1596,/content/хорошо/eu.dac204d4-b6fe-4c79-b60a-868...,0


In [ ]:
def get_duration_sec(path, target_sr=None):
    x, sr = librosa.load(path, sr=target_sr, mono=True)
    return len(x) / sr

df["duration_sec"] = df["path"].apply(lambda p: get_duration_sec(p, target_sr=None))
df.groupby("label")["duration_sec"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1870.0,8.887580,4.480745,2.280000,5.98,7.86,10.64,44.10
1,905.0,13.374618,7.365777,0.529563,8.48,11.82,16.24,68.66


In [ ]:
bad_rows = df[df['duration_sec']==0]

df.drop(bad_rows.index, inplace=True)

In [ ]:
def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        score = model.decision_function(X)
        score = (score - score.min()) / (score.max() - score.min() + 1e-8)
        return score
    return None

def add_row(rows, name, y_true, y_pred_bad, y_score_bad=None):
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred_bad),
        "F1_bad": f1_score(y_true, y_pred_bad, average="binary", pos_label=1),
        "F1_macro": f1_score(y_true, y_pred_bad, average="macro"),
        "Precision_bad": precision_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0),
        "Recall_bad": recall_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0),
        "Recall_good": recall_score(y_true, y_pred_bad, average="binary", pos_label=0, zero_division=0),
    }
    if y_score_bad is not None:
        row["ROC_AUC"] = roc_auc_score(y_true, y_score_bad)
        row["AP"] = average_precision_score(y_true, y_score_bad)
    rows.append(row)
    return rows

def print_report(name, y_true, y_pred_bad, y_score_bad):
    acc = accuracy_score(y_true, y_pred_bad)
    f1_bad = f1_score(y_true, y_pred_bad, average="binary", pos_label=1)
    f1_mac = f1_score(y_true, y_pred_bad, average="macro")
    prec_bad = precision_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0)
    rec_bad = recall_score(y_true, y_pred_bad, average="binary", pos_label=1, zero_division=0)
    auc = roc_auc_score(y_true, y_score_bad)
    ap = average_precision_score(y_true, y_score_bad)

    print(f"[{name}]")
    print(f"  Accuracy      : {acc:.4f}")
    print(f"  F1-bad        : {f1_bad:.4f}")
    print(f"  F1-macro      : {f1_mac:.4f}")
    print(f"  Precision_bad : {prec_bad:.4f}")
    print(f"  Recall_bad    : {rec_bad:.4f}")
    print(f"  ROC-AUC       : {auc:.4f}")
    print(f"  AP            : {ap:.4f}")

In [ ]:
df_train_tab, df_test_tab = train_test_split(
    df, test_size=0.2, random_state=seed, stratify=df["label"].values
)

df_train_base, df_val_base = train_test_split(
    df_train_tab,
    test_size=0.2,
    random_state=seed,
    stratify=df_train_tab["label"].values
)

df_train_base = df_train_base.reset_index(drop=True)
df_val_base   = df_val_base.reset_index(drop=True)
df_test_base  = df_test_tab.reset_index(drop=True)

print("Train:", len(df_train_base), "Val:", len(df_val_base), "Test:", len(df_test_base))
print("Train counts:\n", df_train_base["label"].value_counts())
print("Val counts:\n", df_val_base["label"].value_counts())
print("Test counts:\n", df_test_base["label"].value_counts())

Train: 1776 Val: 444 Test: 555
Train counts:
 label
0    1197
1     579
Name: count, dtype: int64
Val counts:
 label
0    299
1    145
Name: count, dtype: int64
Test counts:
 label
0    374
1    181
Name: count, dtype: int64


In [ ]:
df_test_base

,path,label,duration_sec
0,/content/хорошо/eu.9b808a69-f082-425d-952c-017...,0,6.620000
1,/content/хорошо/eu.ceb83047-6928-4a8a-b19f-080...,0,8.880000
2,/content/хорошо/eu.7ecccdb0-abb6-4966-85f3-0f1...,0,16.600000
3,/content/плохо/eu.ad64d2c8-0510-42c6-9440-9369...,1,6.207625
4,/content/хорошо/eu.21c5bcbe-1f5b-401f-9421-f65...,0,8.880000
...,...,...,...
550,/content/хорошо/eu.0932d188-e28c-44c4-901f-de5...,0,14.140000
551,/content/хорошо/eu.6693d982-e7a8-4d56-b9bc-70e...,0,6.600000
552,/content/хорошо/eu.f01dc51f-f235-46bb-99e4-719...,0,8.580000
553,/content/плохо/eu.fd758618-c194-47cf-82dd-d592...,1,12.060000


## Модели на MFCC-признаках

In [ ]:
SR = 16000
MAX_SECONDS = 10.0
MAX_LEN = int(SR * MAX_SECONDS)

def load_audio(path, sr=SR):
    errors = []
    try:
        x, file_sr = sf.read(path, always_2d=False)
        if x.ndim > 1:
            x = np.mean(x, axis=1)
        x = x.astype(np.float32)
        if file_sr != sr:
            x = librosa.resample(x, orig_sr=file_sr, target_sr=sr)
        return x.astype(np.float32)
    except Exception as e:
        errors.append(f"soundfile: {repr(e)}")

    try:
        x, _ = librosa.load(path, sr=sr, mono=True)
        return x.astype(np.float32)
    except Exception as e:
        errors.append(f"librosa: {repr(e)}")

    raise RuntimeError("Не удалось прочитать аудиофайл: " + " | ".join(errors))

def pad_or_crop(x, max_len=MAX_LEN, train=False):
    if len(x) == 0:
        return np.zeros((max_len,), dtype=np.float32)

    if len(x) > max_len:
        if train:
            start = np.random.randint(0, len(x) - max_len + 1)
        else:
            start = (len(x) - max_len) // 2
        return x[start:start + max_len]

    if len(x) < max_len:
        return np.pad(x, (0, max_len - len(x)), mode="constant")

    return x

In [ ]:
def extract_mfcc_features(path, sr=SR, n_mfcc=13, n_fft=1024, hop_length=256, train=False):
    x = load_audio(path, sr=sr)
    x = pad_or_crop(x, max_len=MAX_LEN, train=train)

    mfcc = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )
    feat = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)], axis=0)
    return feat.astype(np.float32)

def build_Xy_from_df(df_part, n_mfcc=13, train=False):
    X = np.stack([
        extract_mfcc_features(p, n_mfcc=n_mfcc, train=train)
        for p in tqdm(df_part["path"].tolist())
    ])
    y = df_part["label"].astype(int).values
    return X, y

In [ ]:
def tune_threshold_by_f1(y_true, y_score_bad):
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thr = 0.5
    best_f1 = -1.0

    y_true = np.asarray(y_true)

    for thr in thresholds:
        y_pred = (y_score_bad >= thr).astype(int)
        f1 = f1_score(y_true, y_pred, average="binary", pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1

In [ ]:
N_MFCC = 20

X_train, y_train = build_Xy_from_df(df_train_base, n_mfcc=N_MFCC, train=True)
X_val,   y_val   = build_Xy_from_df(df_val_base,   n_mfcc=N_MFCC, train=False)
X_test,  y_test  = build_Xy_from_df(df_test_base,  n_mfcc=N_MFCC, train=False)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

100%|██████████| 555/555 [00:09<00:00, 56.30it/s]

X_train: (1776, 40) X_val: (444, 40) X_test: (555, 40)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

### Логистическая регрессия

In [ ]:
run = start_baseline_run(
    name="mfcc_logreg",
    config={
        "model_family": "baseline",
        "features": "mfcc",
        "classifier": "logreg",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "n_mfcc": N_MFCC,
    },
)

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])

param_grid_lr = [
    {
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs", "liblinear"],
        "clf__C": [0.01, 0.1, 1, 3, 10, 30],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "clf__penalty": ["l1"],
        "clf__solver": ["liblinear"],
        "clf__C": [0.01, 0.1, 1, 3, 10, 30],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "clf__penalty": ["elasticnet"],
        "clf__solver": ["saga"],
        "clf__C": [0.01, 0.1, 1, 3, 10],
        "clf__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "clf__class_weight": [None, "balanced"],
    },
]

gs_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
gs_lr.fit(X_train, y_train)
print("\n[LR] Best CV macro-F1:", gs_lr.best_score_)
print("[LR] Best params:", gs_lr.best_params_)
logreg_tuned = gs_lr.best_estimator_

val_score_bad = get_score_for_auc(logreg_tuned, X_val)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val, val_score_bad)

test_score_bad = get_score_for_auc(logreg_tuned, X_test)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val, val_pred_best, val_score_bad, best_thr)

test_metrics_default = metrics_dict(y_test, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test, test_pred_best, test_score_bad, best_thr)

print("Best threshold (LogReg, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("LogReg tuned (MFCC)", y_test, test_pred_best, test_score_bad)
add_row(rows, "LogReg tuned (MFCC)", y_test, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(gs_lr.best_score_),
        "model/name": "mfcc_logreg",
    },
)

finish_baseline_run(run)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Fitting 5 folds for each of 86 candidates, totalling 430 fits

[LR] Best CV macro-F1: 0.6610261683586273
[LR] Best params: {'clf__C': 1, 'clf__class_weight': 'balanced', 'clf__l1_ratio': 0.5, 'clf__penalty': 'elasticnet', 'clf__solver': 'saga'}
Best threshold (LogReg, bad): 0.53 | Val F1_bad: 0.6116207951070336
[LogReg tuned (MFCC)]
  Accuracy      : 0.6721
  F1-bad        : 0.5260
  F1-macro      : 0.6377
  Precision_bad : 0.4975
  Recall_bad    : 0.5580
  ROC-AUC       : 0.7032
  AP            : 0.5611


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


### SVM

In [ ]:
run = start_baseline_run(
    name="mfcc_svm",
    config={
        "model_family": "baseline",
        "features": "mfcc",
        "classifier": "svm",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "n_mfcc": N_MFCC,
    },
)

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(probability=True))
])

param_grid_svm = [
    {
        "clf__kernel": ["linear"],
        "clf__C": [0.01, 0.1, 1, 3, 10, 30, 100],
        "clf__class_weight": [None, "balanced"],
    },
    {
        "clf__kernel": ["rbf"],
        "clf__C": [0.1, 1, 3, 10, 30, 100],
        "clf__gamma": ["scale", "auto", 0.01, 0.03, 0.1, 0.3, 1.0],
        "clf__class_weight": [None, "balanced"],
    },
]

gs_svm = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_grid_svm,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
gs_svm.fit(X_train, y_train)
print("\n[SVM] Best CV macro-F1:", gs_svm.best_score_)
print("[SVM] Best params:", gs_svm.best_params_)
svm_tuned = gs_svm.best_estimator_

val_score_bad = get_score_for_auc(svm_tuned, X_val)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val, val_score_bad)

test_score_bad = get_score_for_auc(svm_tuned, X_test)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test, test_pred_best, test_score_bad, best_thr)

print("Best threshold (SVM, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("SVM tuned (MFCC)", y_test, test_pred_best, test_score_bad)
add_row(rows, "SVM tuned (MFCC)", y_test, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(gs_svm.best_score_),
        "model/name": "mfcc_svm",
    },
)

finish_baseline_run(run)

Fitting 5 folds for each of 98 candidates, totalling 490 fits

[SVM] Best CV macro-F1: 0.6893219818369082
[SVM] Best params: {'clf__C': 1, 'clf__class_weight': 'balanced', 'clf__gamma': 0.03, 'clf__kernel': 'rbf'}
Best threshold (SVM, bad): 0.4 | Val F1_bad: 0.5953177257525084
[SVM tuned (MFCC)]
  Accuracy      : 0.7045
  F1-bad        : 0.5568
  F1-macro      : 0.6676
  Precision_bad : 0.5450
  Recall_bad    : 0.5691
  ROC-AUC       : 0.7463
  AP            : 0.5850


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


### XGBoost

In [ ]:
run = start_baseline_run(
    name="mfcc_xgboost",
    config={
        "model_family": "baseline",
        "features": "mfcc",
        "classifier": "xgboost",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "n_mfcc": N_MFCC,
    },
)

pipe_xgb = Pipeline([
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=-1
    ))
])

param_dist_xgb = {
    "clf__n_estimators": randint(300, 1200),
    "clf__max_depth": randint(2, 10),
    "clf__learning_rate": uniform(0.01, 0.2),
    "clf__subsample": uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__min_child_weight": randint(1, 10),
    "clf__gamma": uniform(0.0, 0.5)
}

rs_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_dist_xgb,
    n_iter=40,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=seed
)
rs_xgb.fit(X_train, y_train)
print("\n[XGB] Best CV macro-F1:", rs_xgb.best_score_)
print("[XGB] Best params:", rs_xgb.best_params_)
xgb_tuned = rs_xgb.best_estimator_

val_score_bad = get_score_for_auc(xgb_tuned, X_val)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val, val_score_bad)

test_score_bad = get_score_for_auc(xgb_tuned, X_test)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test, test_pred_best, test_score_bad, best_thr)

print("Best threshold (XGB, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("XGBoost tuned (MFCC)", y_test, test_pred_best, test_score_bad)
add_row(rows, "XGBoost tuned (MFCC)", y_test, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(rs_xgb.best_score_),
        "model/name": "mfcc_xgboost",
    },
)

finish_baseline_run(run)

Fitting 5 folds for each of 40 candidates, totalling 200 fits

[XGB] Best CV macro-F1: 0.6704202620124292
[XGB] Best params: {'clf__colsample_bytree': np.float64(0.8654007076432223), 'clf__gamma': np.float64(0.0025307919231093434), 'clf__learning_rate': np.float64(0.04216161028349973), 'clf__max_depth': 3, 'clf__min_child_weight': 2, 'clf__n_estimators': 1121, 'clf__subsample': np.float64(0.7793696571944989)}
Best threshold (XGB, bad): 0.26 | Val F1_bad: 0.5775075987841946
[XGBoost tuned (MFCC)]
  Accuracy      : 0.6721
  F1-bad        : 0.5604
  F1-macro      : 0.6494
  Precision_bad : 0.4979
  Recall_bad    : 0.6409
  ROC-AUC       : 0.7354
  AP            : 0.5798


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


In [ ]:
tab_results = pd.DataFrame(rows).sort_values("F1_bad", ascending=False).reset_index(drop=True)
display(tab_results)

,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
0,SVM tuned (MFCC),0.677419,0.565217,0.654404,0.502146,0.646409,0.692308,0.735107,0.595436
1,LogReg tuned (MFCC),0.607527,0.557576,0.602459,0.439490,0.762431,0.533156,0.708780,0.530343
2,XGBoost tuned (MFCC),0.641577,0.547511,0.625388,0.463602,0.668508,0.628647,0.703665,0.548935


## CNN на Mel-спектрограммах

In [ ]:
df_train_cnn = df_train_base.copy().reset_index(drop=True)
df_val_cnn   = df_val_base.copy().reset_index(drop=True)
df_test_cnn  = df_test_base.copy().reset_index(drop=True)

print("Train:", len(df_train_cnn), "Val:", len(df_val_cnn), "Test:", len(df_test_cnn))
print("Train counts:\n", df_train_cnn["label"].value_counts())
print("Val counts:\n", df_val_cnn["label"].value_counts())
print("Test counts:\n", df_test_cnn["label"].value_counts())

Train: 1776 Val: 444 Test: 555
Train counts:
 label
0    1197
1     579
Name: count, dtype: int64
Val counts:
 label
0    299
1    145
Name: count, dtype: int64
Test counts:
 label
0    374
1    181
Name: count, dtype: int64


In [ ]:
N_MELS = 64
TARGET_T = 256

def mel_spec(x, sr=SR, n_mels=N_MELS, hop_length=256, n_fft=1024):
    S = librosa.feature.melspectrogram(y=x, sr=sr, n_mels=n_mels, hop_length=hop_length, n_fft=n_fft, power=2.0)
    S = librosa.power_to_db(S, ref=np.max)
    return S.astype(np.float32)

def fix_time_dim(S, target_T=TARGET_T):
    T = S.shape[1]
    if T > target_T:
        start = (T - target_T) // 2
        return S[:, start:start + target_T]
    if T < target_T:
        pad = target_T - T
        return np.pad(S, ((0, 0), (0, pad)), mode="constant")
    return S

def delta_feats(S):
    d1 = librosa.feature.delta(S)
    d2 = librosa.feature.delta(S, order=2)
    return d1.astype(np.float32), d2.astype(np.float32)

class SpeechSpecDataset(Dataset):
    def __init__(self, df_part: pd.DataFrame, train=True):
        self.paths = df_part["path"].tolist()
        self.labels = df_part["label"].astype(int).tolist()
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        y = int(self.labels[idx])

        x = load_audio(path, sr=SR)
        x = pad_or_crop(x, max_len=MAX_LEN, train=self.train)

        S = mel_spec(x, sr=SR, n_mels=N_MELS)
        S = fix_time_dim(S, target_T=TARGET_T)

        d1, d2 = delta_feats(S)
        X = np.stack([S, d1, d2], axis=0)

        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, in_ch=3, n_classes=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Dropout(dropout))
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, N_MELS, TARGET_T)
            h = self.net(dummy).view(1, -1).shape[1]
        self.head = nn.Sequential(
            nn.Linear(h, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.head(x)

In [ ]:
def make_class_weights_from_df(df_part, device):
    counts = df_part["label"].value_counts().sort_index()
    n0, n1 = counts.loc[0], counts.loc[1]
    total = n0 + n1
    w0 = total / (2 * n0)
    w1 = total / (2 * n1)
    return torch.tensor([w0, w1], dtype=torch.float32, device=device)

def run_epoch_cnn(model, loader, criterion, device, optimizer=None, train=True):
    if train:
        model.train()
        if optimizer is None:
            raise ValueError("optimizer отсутствует")
    else:
        model.eval()

    losses, ys, preds, probs_bad = [], [], [], []

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            losses.append(loss.item())

            prob_bad = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            pred_labels = (prob_bad >= 0.5).astype(int)

            probs_bad.append(prob_bad)
            ys.extend(yb.detach().cpu().numpy())
            preds.extend(pred_labels)

    probs_bad = np.concatenate(probs_bad) if len(probs_bad) else np.array([])
    ys = np.array(ys)
    preds = np.array(preds)

    f1_bad = f1_score(ys, preds, average="binary", pos_label=1)
    return float(np.mean(losses)), float(f1_bad), ys, preds, probs_bad

In [ ]:
train_ds = SpeechSpecDataset(df_train_cnn, train=True)
val_ds   = SpeechSpecDataset(df_val_cnn, train=False)
test_ds  = SpeechSpecDataset(df_test_cnn, train=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model_cnn = ImprovedCNN(in_ch=3, dropout=0.3).to(device)

class_weights_cnn = make_class_weights_from_df(df_train_cnn, device)
criterion_cnn = nn.CrossEntropyLoss(weight=class_weights_cnn)

optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_cnn = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cnn, mode="max", factor=0.5, patience=2
)

best_val_f1 = -1.0
best_state = None
best_epoch = -1

run = start_baseline_run(
    name="mel_cnn",
    config={
        "model_family": "baseline",
        "features": "mel+delta+delta2",
        "classifier": "cnn",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "epochs": 12,
        "batch_size": 32,
        "n_mels": N_MELS,
        "target_t": TARGET_T,
    },
)

In [ ]:
EPOCHS = 12

for epoch in tqdm(range(1, EPOCHS + 1)):
    tr_loss, tr_f1, *_ = run_epoch_cnn(
        model_cnn, train_loader, criterion_cnn, device,
        optimizer=optimizer_cnn, train=True
    )
    va_loss, va_f1, va_y, va_pred, va_prob_bad = run_epoch_cnn(
        model_cnn, val_loader, criterion_cnn, device,
        optimizer=None, train=False
    )

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1_bad {tr_f1:.3f} | val loss {va_loss:.4f} f1_bad {va_f1:.3f}")
    scheduler_cnn.step(va_f1)

    if run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": float(tr_loss),
            "train/f1_bad": float(tr_f1),
            "val/loss": float(va_loss),
            "val/f1_bad": float(va_f1),
            "lr": float(optimizer_cnn.param_groups[0]["lr"]),
        })

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model_cnn.state_dict().items()}

print("Best val F1_bad:", best_val_f1, "| Best epoch:", best_epoch)

model_cnn.load_state_dict(best_state)
model_cnn.to(device)

va_loss, va_f1, va_y, va_pred_default, va_prob_bad = run_epoch_cnn(
    model_cnn, val_loader, criterion_cnn, device,
    optimizer=None, train=False
)
best_thr_cnn, best_val_f1_cnn = tune_threshold_by_f1(va_y, va_prob_bad)

te_loss, te_f1, te_y, te_pred_default, te_prob_bad = run_epoch_cnn(
    model_cnn, test_loader, criterion_cnn, device,
    optimizer=None, train=False
)

val_pred_default = va_pred_default
test_pred_default = te_pred_default

val_pred_best = (va_prob_bad >= best_thr_cnn).astype(int)

test_pred_best = (te_prob_bad >= best_thr_cnn).astype(int)

val_metrics_default = metrics_dict(va_y, val_pred_default, va_prob_bad, 0.5)
val_metrics_best = metrics_dict(va_y, val_pred_best, va_prob_bad, best_thr_cnn)
test_metrics_default = metrics_dict(te_y, test_pred_default, te_prob_bad, 0.5)
test_metrics_best = metrics_dict(te_y, test_pred_best, te_prob_bad, best_thr_cnn)

print("Best threshold (CNN, bad):", best_thr_cnn, "| Val F1_bad:", best_val_f1_cnn)
print_report("CNN (Mel+delta+delta2)", te_y, test_pred_best, te_prob_bad)
add_row(rows, "CNN (Mel+delta+delta2)", te_y, test_pred_best, te_prob_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr_cnn,
    extra={
        "best/epoch": int(best_epoch),
        "best/val_f1_bad": float(best_val_f1),
        "model/name": "mel_cnn",
    },
)

finish_baseline_run(run)

  8%|▊         | 1/12 [00:47<08:43, 47.63s/it]

Epoch 01 | train loss 1.5456 f1_bad 0.443 | val loss 0.6599 f1_bad 0.509


 17%|█▋        | 2/12 [01:33<07:45, 46.51s/it]

Epoch 02 | train loss 0.6634 f1_bad 0.488 | val loss 0.6356 f1_bad 0.560


 25%|██▌       | 3/12 [02:17<06:49, 45.47s/it]

...

Epoch 11 | train loss 0.6114 f1_bad 0.563 | val loss 0.6371 f1_bad 0.576


100%|██████████| 12/12 [09:16<00:00, 46.39s/it]

Epoch 12 | train loss 0.6047 f1_bad 0.570 | val loss 0.6134 f1_bad 0.587
Best val F1_bad: 0.5866050808314087 | Best epoch: 12


Best threshold (CNN, bad): 0.48000000000000004 | Val F1_bad: 0.5874439461883408
[CNN (Mel+delta+delta2)]
  Accuracy      : 0.5874
  F1-bad        : 0.5588
  F1-macro      : 0.5856
  Precision_bad : 0.4290
  Recall_bad    : 0.8011
  ROC-AUC       : 0.7157
  AP            : 0.5786


best/epoch,▁
best/val_f1_bad,▁
epoch,▁▂▂▃▄▄▅▅▆▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
+27,...


In [ ]:
all_results = pd.DataFrame(rows).sort_values("F1_bad", ascending=False).reset_index(drop=True)
display(all_results)

,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
0,XGBoost tuned (MFCC),0.672072,0.560386,0.649446,0.497854,0.640884,0.687166,0.735353,0.579834
1,CNN (Mel+delta+delta2),0.587387,0.558767,0.585644,0.428994,0.801105,0.483957,0.715684,0.578552
2,SVM tuned (MFCC),0.704505,0.556757,0.667568,0.544974,0.569061,0.770053,0.746300,0.584987
3,LogReg tuned (MFCC),0.672072,0.526042,0.637676,0.497537,0.558011,0.727273,0.703223,0.561069


## Модели на WavLM-эмбеддингах

In [ ]:
from transformers import AutoFeatureExtractor, WavLMModel

MODEL_NAME = "microsoft/wavlm-base"
SR = 16000
MAX_SECONDS = 10.0
MAX_LEN = int(SR * MAX_SECONDS)

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
WavLM = WavLMModel.from_pretrained(MODEL_NAME).to(device)
WavLM.eval()
for p in WavLM.parameters():
    p.requires_grad = False

@torch.no_grad()
def wavlm_embed_batch(paths, batch_size=8, max_len=MAX_LEN, train=False):
    embs = []

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]

        audios = []
        for p in batch_paths:
            x = load_audio(p, sr=SR)
            x = pad_or_crop(x, max_len=max_len, train=train)
            audios.append(x.astype(np.float32))

        inputs = processor(
            audios,
            sampling_rate=SR,
            return_tensors="pt",
            padding=True
        )

        input_values = inputs["input_values"].to(device)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        out = WavLM(input_values=input_values, attention_mask=attention_mask)
        h = out.last_hidden_state

        pooled = h.mean(dim=1)

        embs.append(pooled.cpu().numpy())

    return np.concatenate(embs, axis=0)

X_train_WavLM = wavlm_embed_batch(df_train_base["path"].tolist(), batch_size=8, train=True)
X_val_WavLM   = wavlm_embed_batch(df_val_base["path"].tolist(),   batch_size=8, train=False)
X_test_WavLM  = wavlm_embed_batch(df_test_base["path"].tolist(),  batch_size=8, train=False)

y_train_WavLM = df_train_base["label"].astype(int).values
y_val_WavLM   = df_val_base["label"].astype(int).values
y_test_WavLM  = df_test_base["label"].astype(int).values

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


### XGBoost (WavLM)

In [ ]:
run = start_baseline_run(
    name="wavlm_emb_xgboost",
    config={
        "model_family": "baseline",
        "features": "wavlm_embedding",
        "classifier": "xgboost",
        "encoder_name": MODEL_NAME,
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
    },
)

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=seed,
    n_jobs=-1
)

param_dist_xgb = {
    "n_estimators": randint(300, 1500),
    "max_depth": randint(2, 10),
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "gamma": uniform(0.0, 0.5),
    "reg_alpha": uniform(0.0, 1.0),
    "reg_lambda": uniform(0.5, 2.0),
}

rs_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=40,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=seed
)

rs_xgb.fit(X_train_WavLM, y_train_WavLM)
print("\n[XGB on WavLM emb] Best CV macro-F1:", rs_xgb.best_score_)
print("[XGB on WavLM emb] Best params:", rs_xgb.best_params_)

xgb_WavLM = rs_xgb.best_estimator_

val_score_bad = xgb_WavLM.predict_proba(X_val_WavLM)[:, 1]
best_thr, best_val_f1 = tune_threshold_by_f1(y_val_WavLM, val_score_bad)

test_score_bad = xgb_WavLM.predict_proba(X_test_WavLM)[:, 1]

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val_WavLM, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val_WavLM, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test_WavLM, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test_WavLM, test_pred_best, test_score_bad, best_thr)

print("Best threshold (XGB WavLM, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("XGBoost (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)
add_row(rows, "XGBoost (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(rs_xgb.best_score_),
        "model/name": "wavlm_emb_xgboost",
        "encoder_name": MODEL_NAME,
    },
)

finish_baseline_run(run)

pd.DataFrame(rows).sort_values("F1_bad", ascending=False)

Fitting 5 folds for each of 40 candidates, totalling 200 fits

[XGB on WavLM emb] Best CV macro-F1: 0.7638962461769482
[XGB on WavLM emb] Best params: {'colsample_bytree': np.float64(0.7710164073434198), 'gamma': np.float64(0.012709563372047594), 'learning_rate': np.float64(0.031578285398660894), 'max_depth': 8, 'min_child_weight': 7, 'n_estimators': 863, 'reg_alpha': np.float64(0.5632755719763837), 'reg_lambda': np.float64(1.891032172852255), 'subsample': np.float64(0.6557325817623503)}
Best threshold (XGB WavLM, bad): 0.26 | Val F1_bad: 0.7076923076923077
[XGBoost (WavLM emb)]
  Accuracy      : 0.7820
  F1-bad        : 0.6685
  F1-macro      : 0.7530
  Precision_bad : 0.6630
  Recall_bad    : 0.6740
  ROC-AUC       : 0.8453
  AP            : 0.7789


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
4,XGBoost (WavLM emb),0.781982,0.668493,0.753039,0.663043,0.674033,0.834225,0.845304,0.778875
2,XGBoost tuned (MFCC),0.672072,0.560386,0.649446,0.497854,0.640884,0.687166,0.735353,0.579834
3,CNN (Mel+delta+delta2),0.587387,0.558767,0.585644,0.428994,0.801105,0.483957,0.715684,0.578552
1,SVM tuned (MFCC),0.704505,0.556757,0.667568,0.544974,0.569061,0.770053,0.746300,0.584987
0,LogReg tuned (MFCC),0.672072,0.526042,0.637676,0.497537,0.558011,0.727273,0.703223,0.561069


## SVM (WavLM)

In [ ]:
run = start_baseline_run(
    name="wavlm_emb_svm",
    config={
        "model_family": "baseline",
        "features": "wavlm_embedding",
        "classifier": "svm",
        "encoder_name": MODEL_NAME,
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
    },
)

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(probability=True))
])

param_grid_svm = [
    {"clf__kernel":["linear"], "clf__C":[0.1, 1, 3, 10, 30],
     "clf__class_weight":[None, "balanced"]},
    {"clf__kernel":["rbf"], "clf__C":[0.1, 1, 3, 10, 30],
     "clf__gamma":["scale", "auto", 0.01, 0.03, 0.1],
     "clf__class_weight":[None, "balanced"]}
]

gs_svm = GridSearchCV(
    pipe_svm,
    param_grid_svm,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
gs_svm.fit(X_train_WavLM, y_train_WavLM)

print("\n[SVM on WavLM emb] Best CV macro-F1:", gs_svm.best_score_)
print("[SVM on WavLM emb] Best params:", gs_svm.best_params_)

svm_WavLM = gs_svm.best_estimator_

val_score_bad = get_score_for_auc(svm_WavLM, X_val_WavLM)
best_thr, best_val_f1 = tune_threshold_by_f1(y_val_WavLM, val_score_bad)

test_score_bad = get_score_for_auc(svm_WavLM, X_test_WavLM)

val_pred_default = (val_score_bad >= 0.5).astype(int)
test_pred_default = (test_score_bad >= 0.5).astype(int)

val_pred_best = (val_score_bad >= best_thr).astype(int)
test_pred_best = (test_score_bad >= best_thr).astype(int)

val_metrics_default = metrics_dict(y_val_WavLM, val_pred_default, val_score_bad, 0.5)
val_metrics_best = metrics_dict(y_val_WavLM, val_pred_best, val_score_bad, best_thr)
test_metrics_default = metrics_dict(y_test_WavLM, test_pred_default, test_score_bad, 0.5)
test_metrics_best = metrics_dict(y_test_WavLM, test_pred_best, test_score_bad, best_thr)

print("Best threshold (SVM WavLM, bad):", best_thr, "| Val F1_bad:", best_val_f1)
print_report("SVM (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)
add_row(rows, "SVM (WavLM emb)", y_test_WavLM, test_pred_best, test_score_bad)

log_baseline_metrics(
    run=run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr,
    extra={
        "cv/best_score": float(gs_svm.best_score_),
        "model/name": "wavlm_emb_svm",
        "encoder_name": MODEL_NAME,
    },
)

finish_baseline_run(run)

pd.DataFrame(rows).sort_values("F1_bad", ascending=False)

Fitting 5 folds for each of 60 candidates, totalling 300 fits

[SVM on WavLM emb] Best CV macro-F1: 0.7643779030195639
[SVM on WavLM emb] Best params: {'clf__C': 3, 'clf__class_weight': 'balanced', 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
Best threshold (SVM WavLM, bad): 0.39 | Val F1_bad: 0.7084639498432602
[SVM (WavLM emb)]
  Accuracy      : 0.7784
  F1-bad        : 0.6667
  F1-macro      : 0.7503
  Precision_bad : 0.6543
  Recall_bad    : 0.6796
  ROC-AUC       : 0.8356
  AP            : 0.7533


cv/best_score,▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
test/best_f1_from_val_f1_macro,▁
test/best_f1_from_val_precision,▁
test/best_f1_from_val_recall,▁
test/default_acc,▁
test/default_ap,▁
+20,...


,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
4,XGBoost (WavLM emb),0.781982,0.668493,0.753039,0.663043,0.674033,0.834225,0.845304,0.778875
5,SVM (WavLM emb),0.778378,0.666667,0.750337,0.654255,0.679558,0.826203,0.835554,0.753307
2,XGBoost tuned (MFCC),0.672072,0.560386,0.649446,0.497854,0.640884,0.687166,0.735353,0.579834
3,CNN (Mel+delta+delta2),0.587387,0.558767,0.585644,0.428994,0.801105,0.483957,0.715684,0.578552
1,SVM tuned (MFCC),0.704505,0.556757,0.667568,0.544974,0.569061,0.770053,0.746300,0.584987
0,LogReg tuned (MFCC),0.672072,0.526042,0.637676,0.497537,0.558011,0.727273,0.703223,0.561069


### WavLM бинарная классификация

In [ ]:
from transformers import AutoModel
from copy import deepcopy
import torchaudio

In [ ]:
class RawAudioDataset(Dataset):
    def __init__(self, df, sr=16000, max_seconds=10.0):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.max_len = int(sr * max_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        label = int(self.df.loc[idx, "label"])

        wav, orig_sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if orig_sr != self.sr:
            wav = torchaudio.functional.resample(wav, orig_sr, self.sr)

        wav = wav.squeeze(0)

        if wav.numel() < self.max_len:
            pad = self.max_len - wav.numel()
            wav = torch.nn.functional.pad(wav, (0, pad))
        else:
            wav = wav[:self.max_len]

        attn_mask = (wav.abs() > 1e-8).long()

        return {
            "audio": wav,
            "attention_mask": attn_mask,
            "label": torch.tensor(label, dtype=torch.long),
        }

In [ ]:
class WavLMBinaryClassifier(nn.Module):
    def __init__(self, encoder_name="microsoft/wavlm-base", dropout=0.2, freeze_encoder=True):
        super().__init__()
        self.encoder_name = encoder_name
        self.encoder = AutoModel.from_pretrained(encoder_name)

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, audio, attention_mask=None):
        out = self.encoder(input_values=audio, attention_mask=attention_mask)
        h = out.last_hidden_state

        if attention_mask is not None and hasattr(self.encoder, "_get_feature_vector_attention_mask"):
            feat_mask = self.encoder._get_feature_vector_attention_mask(
                h.shape[1], attention_mask
            )
            feat_mask = feat_mask.unsqueeze(-1).float()
            h = h * feat_mask
            emb = h.sum(dim=1) / feat_mask.sum(dim=1).clamp(min=1e-6)
        else:
            emb = h.mean(dim=1)

        logits = self.head(emb)
        return logits

In [ ]:
def unfreeze_last_n_layers(model, n_last=4):
    for p in model.encoder.parameters():
        p.requires_grad = False

    if n_last <= 0:
        return

    if hasattr(model.encoder, "encoder") and hasattr(model.encoder.encoder, "layers"):
        layers = model.encoder.encoder.layers
    elif hasattr(model.encoder, "layers"):
        layers = model.encoder.layers
    else:
        raise ValueError("Cannot find encoder layers")

    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad = True

In [ ]:
wavlm_bin_train_ds = RawAudioDataset(df_train_base, sr=SR, max_seconds=MAX_SECONDS)
wavlm_bin_val_ds   = RawAudioDataset(df_val_base, sr=SR, max_seconds=MAX_SECONDS)
wavlm_bin_test_ds  = RawAudioDataset(df_test_base, sr=SR, max_seconds=MAX_SECONDS)

wavlm_bin_train_loader = DataLoader(wavlm_bin_train_ds, batch_size=8, shuffle=True, num_workers=2)
wavlm_bin_val_loader   = DataLoader(wavlm_bin_val_ds, batch_size=8, shuffle=False, num_workers=2)
wavlm_bin_test_loader  = DataLoader(wavlm_bin_test_ds, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
def run_epoch_binary(model, loader, criterion, device, optimizer=None, train=False):
    model.train(train)

    all_y = []
    all_pred = []
    all_prob_bad = []
    total_loss = 0.0
    n = 0

    for batch in loader:
        audio = batch["audio"].to(device)
        attn = batch["attention_mask"].to(device)
        y = batch["label"].to(device)

        with torch.set_grad_enabled(train):
            logits = model(audio, attention_mask=attn)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        prob_bad = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = (prob_bad >= 0.5).astype(int)

        all_y.append(y.detach().cpu().numpy())
        all_pred.append(pred)
        all_prob_bad.append(prob_bad)

        bs = y.size(0)
        total_loss += loss.item() * bs
        n += bs

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    y_prob_bad = np.concatenate(all_prob_bad)

    avg_loss = total_loss / max(n, 1)
    f1_bad = f1_score(y_true, y_pred, average="binary", pos_label=1)

    return avg_loss, f1_bad, y_true, y_pred, y_prob_bad

In [ ]:
wavlm_bin_run = start_baseline_run(
    name="wavlm_binary",
    config={
        "model_family": "baseline",
        "features": "raw_audio",
        "classifier": "wavlm_binary_head",
        "encoder_name": "microsoft/wavlm-base",
        "sr": SR,
        "max_seconds": MAX_SECONDS,
        "random_state": seed,
        "epochs": 12,
        "batch_size": 8,
        "freeze_mode": "unfreeze_last_n",
        "unfreeze_last_n": 4,
    },
)

wavlm_bin_model = WavLMBinaryClassifier(
    encoder_name="microsoft/wavlm-base",
    dropout=0.2,
    freeze_encoder=True,
).to(device)

unfreeze_last_n_layers(wavlm_bin_model, n_last=4)

criterion_bin = nn.CrossEntropyLoss()

head_params = list(wavlm_bin_model.head.parameters())
enc_params = [p for p in wavlm_bin_model.encoder.parameters() if p.requires_grad]

optimizer_bin = torch.optim.AdamW(
    [
        {"params": head_params, "lr": 3e-4},
        {"params": enc_params, "lr": 1e-5},
    ],
    weight_decay=1e-2,
)

scheduler_bin = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_bin,
    mode="min",
    factor=0.5,
    patience=1,
)

best_val_loss = float("inf")
best_epoch = -1
best_state = None

EPOCHS = 12

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, *_ = run_epoch_binary(
        wavlm_bin_model,
        wavlm_bin_train_loader,
        criterion_bin,
        device,
        optimizer=optimizer_bin,
        train=True,
    )

    va_loss, va_f1, va_y, va_pred_default, va_prob_bad = run_epoch_binary(
        wavlm_bin_model,
        wavlm_bin_val_loader,
        criterion_bin,
        device,
        optimizer=None,
        train=False,
    )

    scheduler_bin.step(va_loss)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1_bad {tr_f1:.3f} | val loss {va_loss:.4f} f1_bad {va_f1:.3f}")

    if wavlm_bin_run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": float(tr_loss),
            "train/f1_bad": float(tr_f1),
            "val/loss": float(va_loss),
            "val/f1_bad": float(va_f1),
            "lr/head": float(optimizer_bin.param_groups[0]["lr"]),
            "lr/encoder": float(optimizer_bin.param_groups[1]["lr"]),
        })

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_epoch = epoch
        best_state = deepcopy(wavlm_bin_model.state_dict())

print("Best epoch:", best_epoch, "| Best val loss:", best_val_loss)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 01 | train loss 0.5449 f1_bad 0.466 | val loss 0.4426 f1_bad 0.654


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 02 | train loss 0.4604 f1_bad 0.639 | val loss 0.4146 f1_bad 0.687


...

Epoch 10 | train loss 0.3031 f1_bad 0.825 | val loss 0.5003 f1_bad 0.702


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 11 | train loss 0.2874 f1_bad 0.835 | val loss 0.5213 f1_bad 0.703


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch 12 | train loss 0.2953 f1_bad 0.822 | val loss 0.5135 f1_bad 0.711
Best epoch: 2 | Best val loss: 0.41456497816352156


In [ ]:
wavlm_bin_model.load_state_dict(best_state)

va_loss, va_f1, va_y, va_pred_default, va_prob_bad = run_epoch_binary(
    wavlm_bin_model,
    wavlm_bin_val_loader,
    criterion_bin,
    device,
    optimizer=None,
    train=False,
)

best_thr_wavlm_bin, best_val_f1_wavlm_bin = tune_threshold_by_f1(va_y, va_prob_bad)

te_loss, te_f1, te_y, te_pred_default, te_prob_bad = run_epoch_binary(
    wavlm_bin_model,
    wavlm_bin_test_loader,
    criterion_bin,
    device,
    optimizer=None,
    train=False,
)

val_pred_default = (va_prob_bad >= 0.5).astype(int)
test_pred_default = (te_prob_bad >= 0.5).astype(int)

val_pred_best = (va_prob_bad >= best_thr_wavlm_bin).astype(int)
test_pred_best = (te_prob_bad >= best_thr_wavlm_bin).astype(int)

val_metrics_default = metrics_dict(va_y, val_pred_default, va_prob_bad, 0.5)
val_metrics_best = metrics_dict(va_y, val_pred_best, va_prob_bad, best_thr_wavlm_bin)

test_metrics_default = metrics_dict(te_y, test_pred_default, te_prob_bad, 0.5)
test_metrics_best = metrics_dict(te_y, test_pred_best, te_prob_bad, best_thr_wavlm_bin)

print("Best threshold (WavLM binary):", best_thr_wavlm_bin, "| Val F1_bad:", best_val_f1_wavlm_bin)
print_report("WavLM binary classifier", te_y, test_pred_best, te_prob_bad)
add_row(rows, "WavLM binary classifier", te_y, test_pred_best, te_prob_bad)

log_baseline_metrics(
    run=wavlm_bin_run,
    val_default=val_metrics_default,
    val_best=val_metrics_best,
    test_default=test_metrics_default,
    test_best=test_metrics_best,
    thr_bad=best_thr_wavlm_bin,
    extra={
        "best/epoch": int(best_epoch),
        "best/val_loss": float(best_val_loss),
        "best/val_f1_bad": float(best_val_f1_wavlm_bin),
        "model/name": "wavlm_binary_classifier",
        "encoder_name": "microsoft/wavlm-base",
    },
)

finish_baseline_run(wavlm_bin_run)

pd.DataFrame(rows).sort_values("F1_bad", ascending=False)

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Best threshold (WavLM binary): 0.35000000000000003 | Val F1_bad: 0.6956521739130435
[WavLM binary classifier]
  Accuracy      : 0.7856
  F1-bad        : 0.6426
  F1-macro      : 0.7447
  Precision_bad : 0.7039
  Recall_bad    : 0.5912
  ROC-AUC       : 0.8456
  AP            : 0.7491


best/epoch,▁
best/val_f1_bad,▁
best/val_loss,▁
epoch,▁▂▂▃▄▄▅▅▆▇▇█
lr/encoder,███▄▄▃▃▂▂▁▁▁
lr/head,███▄▄▃▃▂▂▁▁▁
test/best_f1_from_val_acc,▁
test/best_f1_from_val_ap,▁
test/best_f1_from_val_auc,▁
test/best_f1_from_val_f1,▁
+29,...


,Model,Accuracy,F1_bad,F1_macro,Precision_bad,Recall_bad,Recall_good,ROC_AUC,AP
4,XGBoost (WavLM emb),0.781982,0.668493,0.753039,0.663043,0.674033,0.834225,0.845304,0.778875
5,SVM (WavLM emb),0.778378,0.666667,0.750337,0.654255,0.679558,0.826203,0.835554,0.753307
6,WavLM binary classifier,0.785586,0.642643,0.744745,0.703947,0.591160,0.879679,0.845599,0.749110
2,XGBoost tuned (MFCC),0.672072,0.560386,0.649446,0.497854,0.640884,0.687166,0.735353,0.579834
3,CNN (Mel+delta+delta2),0.587387,0.558767,0.585644,0.428994,0.801105,0.483957,0.715684,0.578552
1,SVM tuned (MFCC),0.704505,0.556757,0.667568,0.544974,0.569061,0.770053,0.746300,0.584987
0,LogReg tuned (MFCC),0.672072,0.526042,0.637676,0.497537,0.558011,0.727273,0.703223,0.561069
